In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import h5py
import json
from spatialdata_io.readers.visium_hd import _get_affine
from spatialdata.transformations import get_transformation
import numpy as np
import pandas as pd

## fiducials-image-registration.json

In [2]:
with open(r'E:\2024-07-UTBenchmark-SpC\data\raw\Visium_HD\1\SCA002_reseq\IMAGES\SCA002-fiducials-image-registration.json') as file:
    loupe_json = json.load(file)
loupe_json

{'oligo': '...',
 'spot_metadata': {'bitfield_ordering': 'row-major',
  'bitfield_bit_convention': 'top-left',
  'bitfield_encoding': 'base64',
  'bin_level': 8},
 'metadata': {'version': '1.0.0', 'slide_design': 'visium_hd_rc1'},
 'slide_layout_file': 'eyJzbGlkZV91aWQiOiAiSDEtNjI2QlIyWiIsICJmaWxlX2Zvcm1hdCI6ICIxLjAiLCAiYWxpZ25lcl92ZXJzaW9uIjogIjEuMCIsICJpbnB1dF9oYXNoIjogImVlYWNhZjE1ZTg2YjZhNGJhYjhlNWY1OWMwZDRlMzJiIiwgInNsaWRlX2Rlc2lnbiI6ICJ2aXNpdW1faGRfcmMxIiwgImNhcHR1cmVfYXJlYXMiOiB7IkExIjogWzEuMDAwMDEyMzM0MDY1MDM5MywgMC4wMDIyMDg2NDYxMDI0OTA3NjE0LCAyOC4zODgwMTg2Mjc2NzUyMywgLTAuMDAyMTQ3NDk0ODA4ODUxMzA3OCwgMS4wMDAwMzk1OTY2MzYxNTA2LCAyOC4xODMwNTAyMjAxNzkzMiwgMy41NzQyOTg2OTU0NDYxNTQ4ZS0wOSwgLTMuNjA0NzcwNDUyNjYyMzE4NWUtMTAsIDEuMF0sICJEMSI6IFswLjk5OTk5MDcxNzIzNTU4OSwgMC4wMDEzODkzNTM2MzE1NzQ1ODksIDExLjk3OTkwNzQ2MTM5OTA0OCwgLTAuMDAxMzQ3MzIzNDIwOTc0ODQ2NCwgMC45OTk5OTE0NTUwMzU2NzMxLCAxNi45NjU3ODg4NDgyMTY2NTQsIDUuODg4MjY2Mzc4OTkzMjA0ZS0wOSwgLTUuODA1MjkyODY1NDM5NjA5NWUtMTEsIDEuMF19fQ==',
 'spot_

In [3]:
loupe_transform = _get_affine(loupe_json['transform']) # I am not sure what this transformation does
loupe_transform

Affine (x, y -> x, y)
    [2.13558793e-01 9.62939435e-04 3.57938938e+02]
    [-9.40973181e-04  2.13349012e-01  8.15495362e+02]
    [0. 0. 1.]

In [4]:
loupe_transformImages = _get_affine(loupe_json['cytAssistInfo']['transformImages']) # This is the affine transformation matrix to go from one image to the other
loupe_transformImages

Affine (x, y -> x, y)
    [-4.69506676e-02  1.98883381e-03  2.11056525e+03]
    [1.98226082e-03 4.69047221e-02 5.75142090e+02]
    [0. 0. 1.]

## feature_slice.h5

In [6]:
with h5py.File(r"E:\2024-07-UTBenchmark-SpC\data\raw\Visium_HD\1\SCA002_reseq\COUNTS\outs\feature_slice.h5", "r") as f5:
    metadata = json.loads(dict(f5.attrs)["metadata_json"])
metadata

{'sample_id': 'SCA002_reseq',
 'sample_desc': '',
 'slide_name': 'visium_hd_v1',
 'nrows': 3350,
 'ncols': 3350,
 'spot_pitch': 2.0,
 'hd_layout_json': '{"slide_uid":"H1-626BR2Z","file_format":"1.0","aligner_version":"1.0","input_hash":"eeacaf15e86b6a4bab8e5f59c0d4e32b","slide_design":"visium_hd_rc1","transform":[1.0000123340650393,0.0022086461024907614,28.38801862767523,-0.002147494808851308,1.0000395966361506,28.18305022017932,3.574298695446154e-9,-3.6047704526623175e-10,1.0]}',
 'transform_matrices': {'spot_colrow_to_microscope_colrow': [[-9.077802205185833,
    0.32345516091600923,
    37354.76369104768],
   [0.32405343277047594, 9.078336520958153, 3667.3942514900446],
   [7.1574907283966514e-09, -7.184835516759713e-10, 1.0]],
  'microscope_colrow_to_spot_colrow': [[-0.11001892418420935,
    0.0039202260445032385,
    4095.3538999842776],
   [0.003926833037976163, 0.11001565713494478, -550.1567087383947],
   [7.902807947446162e-10, 5.098545851153002e-11, 1.0]],
  'spot_colrow_to_cy

In [7]:
spot_colrow_to_microscope_colrow = _get_affine(metadata['transform_matrices']['spot_colrow_to_microscope_colrow'])
spot_colrow_to_microscope_colrow

Affine (x, y -> x, y)
    [-9.07780221e+00  3.23455161e-01  3.73547637e+04]
    [3.24053433e-01 9.07833652e+00 3.66739425e+03]
    [0. 0. 1.]

In [8]:
microscope_colrow_to_spot_colrow = _get_affine(metadata['transform_matrices']['microscope_colrow_to_spot_colrow'])
microscope_colrow_to_spot_colrow

Affine (x, y -> x, y)
    [-1.10018924e-01  3.92022604e-03  4.09535390e+03]
    [ 3.92683304e-03  1.10015657e-01 -5.50156709e+02]
    [0. 0. 1.]

In [9]:
spot_colrow_to_cytassist_colrow = _get_affine(metadata['transform_matrices']['spot_colrow_to_cytassist_colrow'])
spot_colrow_to_cytassist_colrow

Affine (x, y -> x, y)
    [4.27121248e-01 2.86904847e-03 3.64243558e+02]
    [-2.79247153e-03  4.26710147e-01  8.21693379e+02]
    [0. 0. 1.]

In [10]:
cytassist_colrow_to_spot_colrow = _get_affine(metadata['transform_matrices']['cytassist_colrow_to_spot_colrow'])
cytassist_colrow_to_spot_colrow

Affine (x, y -> x, y)
    [ 2.34171450e+00 -1.59901416e-02 -8.39815428e+02]
    [ 1.59300628e-02  2.34313815e+00 -1.93114353e+03]
    [0. 0. 1.]

In [11]:
hd_layout = json.loads(metadata['hd_layout_json']) # I am not sure what this transformation does
hd_layout

{'slide_uid': 'H1-626BR2Z',
 'file_format': '1.0',
 'aligner_version': '1.0',
 'input_hash': 'eeacaf15e86b6a4bab8e5f59c0d4e32b',
 'slide_design': 'visium_hd_rc1',
 'transform': [1.0000123340650393,
  0.0022086461024907614,
  28.38801862767523,
  -0.002147494808851308,
  1.0000395966361506,
  28.18305022017932,
  3.574298695446154e-09,
  -3.6047704526623175e-10,
  1.0]}

In [12]:
hd_layout_transform = _get_affine(hd_layout['transform']) # I am not sure what this transformation does
hd_layout_transform

Affine (x, y -> x, y)
    [1.00001233e+00 2.20864610e-03 2.83880186e+01]
    [-2.14749481e-03  1.00003960e+00  2.81830502e+01]
    [0. 0. 1.]

## scalefactors_json.json

scalefactors_json.json: The purpose of the file is to record the relative scales of the user-supplied image, the images in the spatial outs, and the Visium array. This file may contain the following fields:
- regist_target_img_scalef: A scaling factor that converts pixel positions in the original, full-resolution microscope image to the downsampled version used in image registration.
- tissue_hires_scalef: A scaling factor that converts pixel positions in the original, full-resolution image to pixel positions in tissue_hires_image.png.
- tissue_lowres_scalef: A scaling factor that converts pixel positions in the original, full-resolution image to pixel positions in tissue_lowres_image.png.
- fiducial_diameter_fullres: The number of pixels that span the diameter of a fiducial spot in the original, full-resolution image.
- spot_diameter_fullres: The number of pixels that span the diameter of a spot in the original, full-resolution image. This field is intended for visualization purposes, and can vary for different slide designs, ranging from 60-70 µm. Spot diameters and spot positions are estimates. It is better to use the known pixel dimensions of a calibrated microscope rather than trying to infer pixel dimensions from the spot diameter.


In [13]:
with open(r'E:\2024-07-UTBenchmark-SpC\data\raw\Visium_HD\1\SCA002_reseq\COUNTS\outs\binned_outputs\square_002um\spatial\scalefactors_json.json') as file:
    scalefactors_002um = json.load(file)
scalefactors_002um    

{'spot_diameter_fullres': 9.087007285917174,
 'bin_size_um': 2.0,
 'microns_per_pixel': 0.22009446422471257,
 'regist_target_img_scalef': 0.121971056,
 'tissue_lowres_scalef': 0.012197105,
 'fiducial_diameter_fullres': 1499.3562021763337,
 'tissue_hires_scalef': 0.121971056}

In [14]:
with open(r'E:\2024-07-UTBenchmark-SpC\data\raw\Visium_HD\1\SCA002_reseq\COUNTS\outs\binned_outputs\square_008um\spatial\scalefactors_json.json') as file:
    scalefactors_008um = json.load(file)
scalefactors_008um    

{'spot_diameter_fullres': 36.3480291436687,
 'bin_size_um': 8.0,
 'microns_per_pixel': 0.22009446422471257,
 'regist_target_img_scalef': 0.121971056,
 'tissue_lowres_scalef': 0.012197105,
 'fiducial_diameter_fullres': 1499.3562021763337,
 'tissue_hires_scalef': 0.121971056}

In [15]:
with open(r'E:\2024-07-UTBenchmark-SpC\data\raw\Visium_HD\1\SCA002_reseq\COUNTS\outs\binned_outputs\square_016um\spatial\scalefactors_json.json') as file:
    scalefactors_016um = json.load(file)
scalefactors_016um    

{'spot_diameter_fullres': 72.6960582873374,
 'bin_size_um': 16.0,
 'microns_per_pixel': 0.22009446422471257,
 'regist_target_img_scalef': 0.121971056,
 'tissue_lowres_scalef': 0.012197105,
 'fiducial_diameter_fullres': 1499.3562021763337,
 'tissue_hires_scalef': 0.121971056}

## tissue_positions.parquet

- barcode: The sequence of the barcode associated to the spot.
- in_tissue: Binary, indicating if the spot falls inside (1) or outside (0) of tissue.
- array_row: the row coordinate of the spot in the array 
- array_col: The column coordinate of the spot in the array. 
- pxl_row_in_fullres: The row pixel coordinate of the center of the spot in the full resolution image.
- pxl_col_in_fullres: The column pixel coordinate of the center of the spot in the full resolution image. pxl_row_in_fullres and pxl_col_in_fullres can be multiplied by the scale factors in scalefactors_json.json to get pixel coordinates in the lower resolution images.


In [26]:
import pandas as pd
pos_002 = pd.read_parquet(r'E:\2024-07-UTBenchmark-SpC\data\raw\Visium_HD\1\SCA002_reseq\COUNTS\outs\binned_outputs\square_002um\spatial\tissue_positions.parquet')
pos_002

,barcode,in_tissue,array_row,array_col,pxl_row_in_fullres,pxl_col_in_fullres
0,s_002um_00000_00000-1,1,0,0,3667.394251,37354.763691
1,s_002um_00000_00001-1,1,0,1,3667.718279,37345.685622
2,s_002um_00000_00002-1,1,0,2,3668.042306,37336.607552
3,s_002um_00000_00003-1,1,0,3,3668.366333,37327.529483
4,s_002um_00000_00004-1,1,0,4,3668.690360,37318.451414
...,...,...,...,...,...,...
11222495,s_002um_03349_03345-1,0,3349,3345,35153.944931,8072.592800
11222496,s_002um_03349_03346-1,0,3349,3346,35154.268726,8063.515136
11222497,s_002um_03349_03347-1,0,3349,3347,35154.592521,8054.437472
11222498,s_002um_03349_03348-1,0,3349,3348,35154.916316,8045.359808


In [17]:
pos_008 = pd.read_parquet(r'E:\2024-07-UTBenchmark-SpC\data\raw\Visium_HD\1\SCA002_reseq\COUNTS\outs\binned_outputs\square_008um\spatial\tissue_positions.parquet')
pos_008

,barcode,in_tissue,array_row,array_col,pxl_row_in_fullres,pxl_col_in_fullres
0,s_008um_00000_00000-1,1,0,0,3681.497801,37341.631810
1,s_008um_00000_00001-1,1,0,1,3682.793909,37305.319533
2,s_008um_00000_00002-1,1,0,2,3684.090018,37269.007258
3,s_008um_00000_00003-1,1,0,3,3685.386127,37232.694984
4,s_008um_00000_00004-1,1,0,4,3686.682235,37196.382713
...,...,...,...,...,...,...
702239,s_008um_00837_00833-1,0,837,833,35154.760980,8177.147115
702240,s_008um_00837_00834-1,0,837,834,35156.056160,8140.836451
702241,s_008um_00837_00835-1,0,837,835,35157.351340,8104.525789
702242,s_008um_00837_00836-1,0,837,836,35158.646519,8068.215130


In [18]:
pos_016 = pd.read_parquet(r'E:\2024-07-UTBenchmark-SpC\data\raw\Visium_HD\1\SCA002_reseq\COUNTS\outs\binned_outputs\square_016um\spatial\tissue_positions.parquet')
pos_016

,barcode,in_tissue,array_row,array_col,pxl_row_in_fullres,pxl_col_in_fullres
0,s_016um_00000_00000-1,1,0,0,3700.302532,37324.122636
1,s_016um_00000_00001-1,1,0,1,3702.894750,37251.498084
2,s_016um_00000_00002-1,1,0,2,3705.486967,37178.873539
3,s_016um_00000_00003-1,1,0,3,3708.079183,37106.249004
4,s_016um_00000_00004-1,1,0,4,3710.671400,37033.624476
...,...,...,...,...,...,...
175556,s_016um_00418_00414-1,0,418,414,35130.777138,8339.897473
175557,s_016um_00418_00415-1,0,418,415,35133.367500,8267.276127
175558,s_016um_00418_00416-1,0,418,416,35135.957862,8194.654790
175559,s_016um_00418_00417-1,0,418,417,35138.548224,8122.033461


## Test transformImages matrix

In [22]:
affine_matrix = np.array(loupe_json['cytAssistInfo']['transformImages'])
coords = np.hstack([pos_002[['pxl_col_in_fullres', 'pxl_row_in_fullres']].values, np.ones((pos_002.shape[0], 1))])
transformed_coords = coords @ affine_matrix.T
df_transformed = pd.DataFrame({
    'x': transformed_coords[:, 0],
    'y': transformed_coords[:, 1],
})
df_transformed.to_csv(r'E:\2024-07-UTBenchmark-SpC\data\raw\Visium_HD\1\SCA002_reseq\test.csv', index=False)

# NOTE: This shows that the affine matrix converts from the fullres image coordinate space to the cytassist image coordinate space.

In [23]:
loupe_transformImages

Affine (x, y -> x, y)
    [-4.69506676e-02  1.98883381e-03  2.11056525e+03]
    [1.98226082e-03 4.69047221e-02 5.75142090e+02]
    [0. 0. 1.]

In [24]:
np.dot(metadata['transform_matrices']['spot_colrow_to_cytassist_colrow'], metadata['transform_matrices']['microscope_colrow_to_spot_colrow'])

array([[-4.69798661e-02,  1.99007066e-03,  2.11187780e+03],
       [ 1.98349359e-03,  4.69338920e-02,  5.75499770e+02],
       [-1.45640443e-08, -1.31302095e-08,  1.00062190e+00]])

## sdata

In [4]:
from sparrow.io._visium_hd import visium_hd

sdata = visium_hd(
    path=r'E:\2024-07-UTBenchmark-SpC\data\raw\Visium_HD\1\SCA002_reseq\COUNTS\outs', 
    bin_size=[2,8,16], 
    dataset_id="", # Had to adapt spatialdata io visium reader according to https://github.com/scverse/spatialdata-io/issues/218
    bins_as_squares=True, 
    fullres_image_file=r'E:\2024-07-UTBenchmark-SpC\data\raw\Visium_HD\1\SCA002_reseq\IMAGES\SCA002_HE_hires.tif', # Adapted harpy visium hd wrapper to take this argument, I also had to set scale_factors=None in line 361
    output= None) # I run into buffer issues if I try to write the sdata to disk
sdata

c:\Users\julienm\miniconda3\envs\harpy\lib\site-packages\anndata\_core\anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
c:\Users\julienm\miniconda3\envs\harpy\lib\site-packages\anndata\_core\anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
c:\Users\julienm\miniconda3\envs\harpy\lib\site-packages\anndata\_core\anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


SpatialData object
├── Images
│     ├── '_cytassist_image': DataArray[cyx] (3, 3000, 3200)
│     ├── '_full_image': DataArray[cyx] (49192, 3, 32447)
│     ├── '_hires_image': DataArray[cyx] (3, 3958, 6000)
│     └── '_lowres_image': DataArray[cyx] (3, 396, 600)
├── Labels
│     ├── '_square_002um_labels': DataArray[yx] (3347, 3350)
│     ├── '_square_008um_labels': DataArray[yx] (837, 838)
│     └── '_square_016um_labels': DataArray[yx] (419, 419)
├── Shapes
│     ├── '_square_002um': GeoDataFrame shape: (7375079, 1) (2D shapes)
│     ├── '_square_008um': GeoDataFrame shape: (462269, 1) (2D shapes)
│     └── '_square_016um': GeoDataFrame shape: (115975, 1) (2D shapes)
└── Tables
      ├── 'square_002um': AnnData (7375079, 19059)
      ├── 'square_008um': AnnData (462269, 19059)
      └── 'square_016um': AnnData (115975, 19059)
with coordinate systems:
    ▸ 'downscaled_hires', with elements:
        _hires_image (Images), _square_002um_labels (Labels), _square_008um_labels (Labels), _s

In [5]:
get_transformation(sdata.images['_full_image'], get_all=True)

{'global': Identity }

In [6]:
get_transformation(sdata.images['_hires_image'], get_all=True)

{'downscaled_hires': Identity }

In [8]:
get_transformation(sdata.images['_lowres_image'], get_all=True)

{'downscaled_lowres': Identity }

In [10]:
get_transformation(sdata.labels['_square_002um_labels'], get_all=True)

{'global': Sequence 
     Sequence 
         Affine (x, y -> x, y)
             [-9.07786461e+00  3.23471549e-01  3.73546649e+04]
             [3.23914091e-01 9.07825662e+00 3.66753013e+03]
             [0. 0. 1.]
     Identity ,
 'downscaled_hires': Sequence 
     Sequence 
         Affine (x, y -> x, y)
             [-9.07786461e+00  3.23471549e-01  3.73546649e+04]
             [3.23914091e-01 9.07825662e+00 3.66753013e+03]
             [0. 0. 1.]
     Scale (x, y)
         [0.12197106 0.12197106],
 'downscaled_lowres': Sequence 
     Sequence 
         Affine (x, y -> x, y)
             [-9.07786461e+00  3.23471549e-01  3.73546649e+04]
             [3.23914091e-01 9.07825662e+00 3.66753013e+03]
             [0. 0. 1.]
     Scale (x, y)
         [0.0121971 0.0121971]}

In [13]:
get_transformation(sdata.shapes['_square_002um'], get_all=True)

{'global': Identity ,
 'downscaled_hires': Scale (x, y)
     [0.12197106 0.12197106],
 'downscaled_lowres': Scale (x, y)
     [0.0121971 0.0121971]}

In [14]:
sdata.coordinate_systems

['downscaled_lowres', 'global', 'downscaled_hires']

In [25]:
sdata.tables['square_002um'].obs

,cell_ID,in_tissue,array_row,array_col,fov_labels
s_002um_02058_00301-1,6894602,1,2058,301,_square_002um_labels
s_002um_00700_02130-1,2347131,1,700,2130,_square_002um_labels
s_002um_02043_00141-1,6844192,1,2043,141,_square_002um_labels
s_002um_02278_00850-1,7632151,1,2278,850,_square_002um_labels
s_002um_00639_00600-1,2141251,1,639,600,_square_002um_labels
...,...,...,...,...,...
s_002um_02017_00654-1,6757605,1,2017,654,_square_002um_labels
s_002um_00305_01021-1,1022772,1,305,1021,_square_002um_labels
s_002um_01817_01607-1,6088558,1,1817,1607,_square_002um_labels
s_002um_02189_01783-1,7334934,1,2189,1783,_square_002um_labels


In [18]:
sdata.tables['square_002um'].obsm['spatial']

array([[35287.99210204, 22448.13572521],
       [18245.19462656, 10712.30570162],
       [36735.62932287, 22260.13750757],
       ...,
       [23354.21544104, 20683.27467739],
       [21876.84092548, 24117.39031592],
       [ 8545.85568371, 24202.15241172]])

In [34]:
pos_002[pos_002['barcode'] == 's_002um_02058_00301-1']

,barcode,in_tissue,array_row,array_col,pxl_row_in_fullres,pxl_col_in_fullres
6894601,s_002um_02058_00301-1,1,2058,301,22448.135725,35287.992102
